# TacticRadar: Exploratory Data Analysis (EDA)
## Lichess Chess Tactical Motif Distribution & Co-occurrence Analysis

This notebook analyzes the **Lichess Puzzle Database** focusing on the 4 primary tactical motifs:
- **Fork**
- **Pin**
- **Skewer**
- **Discovered Attack**

We investigate class distributions, multi-label co-occurrence relationships, rating distributions, and the necessity of negative ('quiet') sampling.

In [ ]:
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.loader import load_raw_dataset
from src.data.preprocess import preprocess_dataset
from src.data.split import split_dataset, LABEL_COLS

### 1. Load Preprocessed & Stratified Dataset

In [ ]:
with open('../configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Ensure relative path from notebooks directory
config['data']['raw_dir'] = '../data/raw'
config['data']['processed_dir'] = '../data/processed'

train_df = pd.read_parquet('../data/processed/train.parquet')
val_df = pd.read_parquet('../data/processed/val.parquet')
test_df = pd.read_parquet('../data/processed/test.parquet')
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f"Total dataset size: {len(df):,}")
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
df.head()

### 2. Label Prevalence & Class Balance

In [ ]:
label_names = ['Fork', 'Pin', 'Skewer', 'Discovered Attack']
counts = [df[col].sum() for col in LABEL_COLS]
percentages = [100.0 * c / len(df) for c in counts]

plt.figure(figsize=(9, 5))
bars = plt.bar(label_names, counts, color=['#3b82f6', '#10b981', '#f59e0b', '#ec4899'])
plt.ylabel('Count of Positions', fontsize=12)
plt.title('Tactical Motif Frequency in Balanced Dataset', fontsize=14, fontweight='bold')
for bar, pct in zip(bars, percentages):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 100, f"{yval:,}\n({pct:.1f}%)", ha='center', va='bottom', fontsize=10)
plt.ylim(0, max(counts) * 1.2)
plt.tight_layout()
plt.show()

### 3. Multi-Label Co-occurrence Matrix
Chess tactics frequently co-occur (e.g. an absolute pin enabling a devastating knight fork or discovered attack).

In [ ]:
Y = df[LABEL_COLS].values
cooc = np.zeros((4, 4), dtype=int)
for i in range(4):
    for j in range(4):
        cooc[i, j] = int(((Y[:, i] == 1) & (Y[:, j] == 1)).sum())

plt.figure(figsize=(8, 6))
sns.heatmap(cooc, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title('Motif Co-occurrence Matrix (Overlapping Tactical Themes)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 4. Puzzle Rating Distribution by Primary Motif

In [ ]:
plt.figure(figsize=(10, 5))
for col, name in zip(LABEL_COLS, label_names):
    sub = df[df[col] == 1]['Rating']
    sns.kdeplot(sub, label=f"{name} (mean: {sub.mean():.0f})", fill=True, alpha=0.15)
plt.title('Lichess Puzzle Rating Distribution by Motif', fontsize=13, fontweight='bold')
plt.xlabel('Rating (Glicko-2)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()